# Score Prioridade MVP

Este notebook explica as features usadas para montar o `score_prioridade_mvp` e mostra, passo a passo, como o score final e a faixa de prioridade são calculados.

Ponto central: este score **não é um target supervisionado de Machine Learning**. Ele é um índice transparente, baseado em regras, para indicar prioridade relativa para investigação e possível investimento em leitos SUS.

## Setup

Primeiro carregamos bibliotecas, localizamos a raiz do projeto e lemos a tabela do score. Se o Parquet ainda não existir, o notebook tenta reconstruí-lo a partir das facts em `data/processed/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "pyproject.toml").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Raiz do projeto não encontrada.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SCORE_PATH = PROCESSED_DIR / "score_prioridade_mvp.parquet"
DIM_PATH = PROCESSED_DIR / "dim_municipio_rs.parquet"

In [ ]:
from src.models.build_score_prioridade_mvp import SCORE_WEIGHTS, build_score_prioridade_mvp

if not SCORE_PATH.exists():
    build_score_prioridade_mvp(output_path=SCORE_PATH)

score = pd.read_parquet(SCORE_PATH)
dim = pd.read_parquet(DIM_PATH)

dim_names = dim.rename(columns={"municipio_id_ibge": "municipio_id"})[
    ["municipio_id", "municipio_nome", "regiao_saude", "macroregiao_saude"]
]

score_named = score.merge(dim_names, on="municipio_id", how="left", validate="many_to_one")
score.head()

## Unidade analítica

Cada linha representa:

```text
município + ano + tipo_leito
```

Os tipos de leito do MVP são mantidos separados:

- `leitos_clinicos`
- `leitos_cirurgicos`
- `leitos_obstetricos`
- `leitos_pediatricos`
- `uti_adulto`

Não existe um tipo agregado chamado `leitos_gerais` neste score.

In [ ]:
coverage = {
    "linhas": len(score),
    "municipios": score["municipio_id"].nunique(),
    "anos": sorted(score["ano"].unique().tolist()),
    "tipos_leito": sorted(score["tipo_leito"].unique().tolist()),
    "chave_duplicada": score.duplicated(["municipio_id", "ano", "tipo_leito"]).any(),
}

coverage

## Fórmula do score

A fórmula implementada é:

```text
score_prioridade_mvp =
  0.30 * demanda_residente_normalizada
+ 0.25 * deficit_oferta_sus_normalizado
+ 0.20 * evasao_hospitalar_normalizada
+ 0.15 * risco_populacional_normalizado
+ 0.10 * crescimento_demanda_normalizado
```

A leitura prática é: o score soma sinais de demanda, baixa oferta SUS, dependência de internações fora do município, perfil populacional e crescimento histórico.

In [ ]:
pd.DataFrame(
    [{"componente": component, "peso": weight} for component, weight in SCORE_WEIGHTS.items()]
).assign(peso_percentual=lambda df: df["peso"] * 100)

## Catálogo das features do score

A tabela abaixo resume o papel de cada coluna importante. O foco é a regra de negócio, não apenas o nome técnico da variável.

In [ ]:
feature_catalog = pd.DataFrame(
    [
        {
            "feature": "populacao_alvo",
            "grupo": "denominador",
            "interpretação": "População usada como base de comparação para o tipo de leito.",
            "como é calculada": "Total para clínicos/cirúrgicos; mulheres em idade fértil para obstétricos; população 0-14 para pediátricos; população adulta proxy para UTI Adulto.",
            "cuidado": "É uma aproximação populacional; não mede necessidade clínica individual.",
        },
        {
            "feature": "internacoes_residentes",
            "grupo": "demanda",
            "interpretação": "Volume de internações SIH de moradores do município.",
            "como é calculada": "Contagem de AIHs agrupadas por município de residência, ano e tipo de leito proxy.",
            "cuidado": "Usa residência, não local do hospital. Isso evita confundir município polo com demanda local.",
        },
        {
            "feature": "demanda_residente_taxa_por_1000_pop_alvo",
            "grupo": "demanda",
            "interpretação": "Demanda residente ajustada pelo tamanho da população-alvo.",
            "como é calculada": "internacoes_residentes / populacao_alvo * 1000.",
            "cuidado": "Taxas pequenas podem oscilar bastante em municípios pequenos.",
        },
        {
            "feature": "leitos_sus_media",
            "grupo": "oferta",
            "interpretação": "Oferta média anual de leitos SUS do tipo no município.",
            "como é calculada": "Média mensal de `QT_SUS` no CNES por município, ano e tipo de leito.",
            "cuidado": "Não mede ocupação diária, fila, qualidade assistencial ou disponibilidade de equipe.",
        },
        {
            "feature": "oferta_sus_por_1000_pop_alvo",
            "grupo": "oferta",
            "interpretação": "Oferta SUS relativa à população-alvo.",
            "como é calculada": "leitos_sus_media / populacao_alvo * 1000.",
            "cuidado": "Um município pode ter baixa oferta local e ainda ser atendido por rede regional próxima.",
        },
        {
            "feature": "benchmark_estadual_leitos_sus_por_1000_pop_alvo",
            "grupo": "oferta",
            "interpretação": "Referência estadual de oferta para o mesmo ano e tipo de leito.",
            "como é calculada": "Soma estadual de leitos SUS / soma estadual da população-alvo * 1000.",
            "cuidado": "É uma régua comparativa simples, não uma meta normativa do SUS.",
        },
        {
            "feature": "deficit_oferta_sus_raw",
            "grupo": "déficit",
            "interpretação": "Sinal de baixa oferta SUS onde também existe demanda residente observada.",
            "como é calculada": "max(benchmark estadual - oferta local, 0) * demanda_residente_taxa_por_1000_pop_alvo.",
            "cuidado": "Ausência de leito não vira prioridade máxima automaticamente; sem demanda residente, o déficit bruto fica zero.",
        },
        {
            "feature": "internacoes_residentes_fora",
            "grupo": "fluxo",
            "interpretação": "Quantidade de internações de moradores realizadas fora do próprio município.",
            "como é calculada": "Contagem de registros em que município de residência é diferente do município de internação.",
            "cuidado": "Pode refletir dependência regional, pactuação da rede ou necessidade de serviço especializado.",
        },
        {
            "feature": "evasao_hospitalar_raw",
            "grupo": "fluxo",
            "interpretação": "Proporção da demanda residente atendida fora do município.",
            "como é calculada": "internacoes_residentes_fora / internacoes_residentes.",
            "cuidado": "Alta evasão não prova falta local de leito; pode ser desenho regional esperado.",
        },
        {
            "feature": "risco_populacional_raw",
            "grupo": "risco",
            "interpretação": "Proxy simples de população mais relacionada ao tipo de leito.",
            "como é calculada": "Idosos para clínicos/cirúrgicos; mulheres em idade fértil para obstétricos; crianças para pediátricos; idosos sobre população adulta para UTI Adulto.",
            "cuidado": "Não substitui indicadores epidemiológicos detalhados.",
        },
        {
            "feature": "crescimento_demanda_raw",
            "grupo": "tendência",
            "interpretação": "Mudança anual da demanda residente.",
            "como é calculada": "internacoes_residentes do ano atual / internações do ano anterior - 1.",
            "cuidado": "Quando não há histórico suficiente, fica ausente e o componente normalizado recebe valor neutro.",
        },
    ]
)

feature_catalog

## Normalização dos componentes

Os componentes brutos têm escalas diferentes. Por isso, cada componente entra no score depois de normalização min-max entre 0 e 1 dentro de cada grupo `ano + tipo_leito`.

Em termos práticos: um município de UTI Adulto em 2024 só é comparado com outros municípios de UTI Adulto em 2024. Isso evita misturar tipos de leito que têm lógicas muito diferentes.

In [ ]:
component_columns = [
    "demanda_residente_normalizada",
    "deficit_oferta_sus_normalizado",
    "evasao_hospitalar_normalizada",
    "risco_populacional_normalizado",
    "crescimento_demanda_normalizado",
    "score_prioridade_mvp",
]

score[component_columns].agg(["min", "max", "mean"]).T

In [ ]:
component_ranges_by_type = (
    score.groupby(["ano", "tipo_leito"])[component_columns[:-1]]
    .agg(["min", "max"])
    .reset_index()
)

component_ranges_by_type.head(10)

## Crescimento de demanda e valor neutro

O componente de crescimento precisa de histórico. Quando o município não tem ano anterior com demanda positiva para aquele tipo de leito, o projeto não inventa crescimento.

Nesses casos:

- `crescimento_demanda_raw` fica ausente;
- `crescimento_demanda_disponivel` fica `False`;
- `crescimento_demanda_normalizado` recebe valor neutro `0.5`.

Isso evita punir ou premiar demais municípios sem série histórica suficiente.

In [ ]:
(
    score.groupby(["ano", "tipo_leito", "crescimento_demanda_disponivel"])
    .size()
    .rename("linhas")
    .reset_index()
    .head(30)
)

## Como a faixa de prioridade é definida

Depois do score contínuo, o projeto cria uma leitura categórica para facilitar comunicação pública:

- `baixa`: score menor que 0.40
- `media`: score de 0.40 até menor que 0.60
- `alta`: score de 0.60 até menor que 0.75
- `muito_alta`: score maior ou igual a 0.75

Essas faixas são pontos de corte iniciais. Elas ajudam a navegar, mas não devem ser lidas como decisão automática de investimento.

In [ ]:
(
    score["faixa_prioridade"]
    .value_counts()
    .rename_axis("faixa_prioridade")
    .reset_index(name="linhas")
)

## Ranking inicial por ano e tipo de leito

A célula abaixo mostra os maiores scores do ano mais recente disponível. A interpretação correta é: municípios com maior prioridade relativa para investigação, não municípios que obrigatoriamente devem receber novos leitos.

In [ ]:
ano_referencia = int(score_named["ano"].max())
ano_referencia

In [ ]:
ranking_columns = [
    "municipio_nome",
    "regiao_saude",
    "tipo_leito",
    "score_prioridade_mvp",
    "faixa_prioridade",
    "internacoes_residentes",
    "internacoes_residentes_fora",
    "demanda_residente_taxa_por_1000_pop_alvo",
    "leitos_sus_media",
    "deficit_oferta_sus_raw",
    "evasao_hospitalar_raw",
]

(
    score_named.query("ano == @ano_referencia")
    .sort_values(["tipo_leito", "score_prioridade_mvp"], ascending=[True, False])
    [ranking_columns]
    .groupby("tipo_leito", group_keys=False)
    .head(10)
)

## Explicando um município

A função abaixo decompõe o score de uma linha em contribuições ponderadas. Essa é a forma mais honesta de explicar um ranking: mostrar quais componentes puxaram o score para cima.

In [ ]:
def explicar_linha_score(row: pd.Series) -> pd.DataFrame:
    rows = []
    for component, weight in SCORE_WEIGHTS.items():
        value = row[component]
        rows.append(
            {
                "componente": component,
                "valor_normalizado": value,
                "peso": weight,
                "contribuicao_no_score": value * weight,
            }
        )
    return pd.DataFrame(rows).sort_values("contribuicao_no_score", ascending=False)


exemplo = (
    score_named.query("ano == @ano_referencia")
    .sort_values("score_prioridade_mvp", ascending=False)
    .iloc[0]
)

exemplo[["municipio_nome", "tipo_leito", "score_prioridade_mvp", "faixa_prioridade"]]

In [ ]:
explicar_linha_score(exemplo)

## Navegação por tipo de leito

Use a função abaixo para navegar por um tipo de leito específico. Isso é útil porque clínico, obstétrico, pediátrico e UTI Adulto não devem ser comparados como se fossem a mesma coisa.

In [ ]:
def mostrar_ranking_tipo(tipo_leito: str, ano: int | None = None, n: int = 15) -> pd.DataFrame:
    ano = ano or ano_referencia
    return (
        score_named.query("tipo_leito == @tipo_leito and ano == @ano")
        .sort_values("score_prioridade_mvp", ascending=False)
        [
            [
                "municipio_nome",
                "regiao_saude",
                "score_prioridade_mvp",
                "faixa_prioridade",
                "demanda_residente_normalizada",
                "deficit_oferta_sus_normalizado",
                "evasao_hospitalar_normalizada",
                "risco_populacional_normalizado",
                "crescimento_demanda_normalizado",
            ]
        ]
        .head(n)
    )


mostrar_ranking_tipo("leitos_clinicos")

In [ ]:
for tipo in sorted(score_named["tipo_leito"].unique()):
    print(f"\n=== {tipo} ===")
    display(mostrar_ranking_tipo(tipo, ano_referencia, n=5))

## Checagens de qualidade

Antes de publicar ranking, sempre valide:

- se a chave `municipio_id + ano + tipo_leito` é única;
- se todos os componentes normalizados estão entre 0 e 1;
- se os tipos de leito esperados estão presentes;
- se as ausências são esperadas, especialmente em crescimento histórico;
- se os resultados fazem sentido quando separados por tipo de leito.

In [ ]:
quality_checks = {
    "chave_unica": not score.duplicated(["municipio_id", "ano", "tipo_leito"]).any(),
    "componentes_minimo_ok": bool((score[component_columns].min() >= 0).all()),
    "componentes_maximo_ok": bool((score[component_columns].max() <= 1).all()),
    "sem_leitos_gerais": "leitos_gerais" not in set(score["tipo_leito"]),
}

quality_checks

In [ ]:
missing = (
    score.isna()
    .sum()
    .rename("valores_ausentes")
    .to_frame()
    .assign(percentual=lambda df: df["valores_ausentes"] / len(score))
)

missing.query("valores_ausentes > 0")

## Limitações metodológicas

Este score é útil para organizar a investigação, mas não substitui decisão técnica do SUS.

Principais limitações:

- o SIH não informa diretamente o tipo de leito ocupado; o projeto usa uma classificação proxy por procedimento, CID, idade e uso de UTI;
- evasão hospitalar pode representar falta local de estrutura, mas também pode refletir pactuação regional legítima;
- `QT_SUS` mede oferta registrada no CNES, não disponibilidade diária real;
- o score não considera custo de implantação, equipe disponível, obras, regulação, qualidade assistencial ou fila real;
- municípios pequenos podem ter taxas instáveis por baixo volume;
- as faixas de prioridade são pontos de corte iniciais para comunicação, não decisão automática.

A frase correta para comunicar resultados é: **alta prioridade relativa para investigação e possível investimento**, sempre com as limitações explícitas.